# A3.3 — Router Şablonlarını Başkalaştırma (Paraphrase) — Colab

Bu notebook **Google Colab'da**, ücretsiz T4 GPU ile çalışacak şekilde yazıldı.

**Ne yapıyor:** Router için üretilen ~24.999 sorunun **hepsini değil**,
yalnızca arkasındaki **~84 benzersiz şablonu** (yer tutucu içeren hâliyle)
büyük bir LLM'e (Qwen2.5-7B-Instruct) verip başkalaştırıyor. Başkalaştırılmış
şablonlar sonra yerelde (GPU'suz, saniyeler içinde) gerçek ürün/kategori/
tedarikçi adlarıyla yeniden çoğaltılıp tam veri setine dönüştürülüyor.

> ⚠️ **İlk tasarım hatası ve düzeltmesi:** İlk denemede 24.999 satırın HER
> BİRİNİ ayrı ayrı modele gönderiyorduk — 100 satır 357 saniye sürmüştü,
> tüm veri seti için ~25 saate karşılık geliyordu (Colab'ın ücretsiz oturum
> sınırı ~12 saat, bu yüzden imkânsızdı). Oysa 24.999 satırın tamamı yalnızca
> ~84 şablonun ürün/kategori/tedarikçi ile doldurulmuş hâli. Şimdi yalnızca
> o 84 şablonu başkalaştırıp, aynı çoğaltmayı yerelde tekrar uyguluyoruz —
> ~250x daha az model çağrısı, ~5 dakika.

> ⚠️ **İkinci tasarım hatası ve düzeltmesi (bu turda):** İlk denemede
> varyantların ~%40'ında model cümlenin **niyetini veya anlam yönünü tersine
> çeviriyordu** — "listesini görebilir miyim?" gibi bir SORU, "listeden
> kaldırılmasını istiyorum" gibi bir KOMUTA dönüşebiliyordu. Sebep: prompt
> yalnızca "anlamı koru" diyordu, cümlenin bir SORU/BİLGİ TALEBİ olduğunu ve
> olumsuzluk/gerekirlik yönünün (ör. "-mesi gereken") değişmemesi gerektiğini
> hiç açıklamıyordu — A3.4'te aynı sınıf hatayı (karar tipinin iş anlamını
> prompt'a yazmayınca ortaya çıkan yön hatası) çözünce buraya da uyarladık.
> Düzeltme iki parçalı: **(1)** prompt'a niyet-koruma kuralı + doğru/yanlış
> örnek eklendi, **(2)** A3.4'teki guard deseniyle aynı fikir — her aday
> parafraz, ikinci bir kısa LLM çağrısıyla ("aynı bilgiyi mi istiyor,
> evet/hayır?") doğrulanıp geçemeyeni atılıyor (bkz. 4. hücre,
> `anlam_korundu_mu`).

**Neden ayrı bir model?** Kişi B'nin kuracağı Qwen2.5-**1.5B** üretimde
(routing + gerekçe yazımı) çalışacak küçük model. Bu notebook'taki
Qwen2.5-**7B** yalnızca **eğitim verisi hazırlarken bir kerelik** kullanılan,
çok daha güçlü bir model.

**Sıra:**
1. Runtime → Değiştir → Donanım hızlandırıcı → **T4 GPU** seç
2. `sablon_listesi.jsonl` dosyasını yükle (yerelde `uv run python -m
   training.build_dataset --sablon-ihrac-et data/egitim/sablon_listesi.jsonl`
   ile üretilir — yalnızca ~84 satır, çok küçük bir dosya)
3. Tüm şablonları tek seferde işle (küçük olduğu için örnekleme adımına
   gerek yok), sonucu indir


## 1. Ortamı hazırla

In [ ]:
!nvidia-smi


In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece


## 2. Şablon dosyasını yükle

Aşağıdaki hücre bir dosya seçme penceresi açacak.

- **Tam tur** (84 şablon, tüm araçlar): `sablon_listesi.jsonl` yükle
  (`uv run python -m training.build_dataset --sablon-ihrac-et data/egitim/sablon_listesi.jsonl`).
- **Hedefli tur** (yalnızca seyrek araçlar, daha yüksek `n` ile — bkz. 5.
  hücredeki `VARYANT_SAYISI`): `sablon_listesi_seyrek.jsonl` yükle
  (`uv run python -m training.build_dataset --sablon-ihrac-et data/egitim/sablon_listesi_seyrek.jsonl --sablon-ihrac-araclar onay_kuyrugu_sorgula genel_stok_durumu_sorgula gecelik_ozet_sorgula`).

24.999 satırlık `router_sorulari.jsonl`'ü **YÜKLEME** — ikisi de küçük şablon
dosyaları (~28-84 satır).

In [ ]:
from google.colab import files

yuklenen = files.upload()
sablon_dosyasi = next(iter(yuklenen.keys()))
print("Yüklenen dosya:", sablon_dosyasi)


## 3. Modeli yükle (4-bit, T4'e sığması için)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ADI = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ADI)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ADI,
    quantization_config=bnb_config,
    device_map="auto",
)
print("Model yüklendi.")


## 4. Başkalaştırma fonksiyonu

Dört güvenlik kontrolü var: **(1)** yer tutucu token (ör. `KATEGORI_ADI`)
korunmamışsa o varyant atılır, **(2)** Çince/Korece/Japonca karaktere
kayarsa atılır, **(3)** numaralandırma/madde işareti kalıntıları temizlenir,
**(4) YENİ** — her aday varyant, aynı modele ikinci bir kısa çağrıyla
("bu iki cümle aynı bilgiyi mi istiyor?") sorulup **anlam/niyet yönü**
doğrulanır; "hayır" derse o varyant atılır. Hiçbiri geçemezse 2 kez tekrar
dener, yine olmazsa o şablon için boş liste döner (bir sonraki hücrede
orijinal şablon korunur, veri kaybı olmaz).

In [ ]:
import re

CJK_DESENI = re.compile(r"[一-鿿぀-ヿ가-힯]")
LATIN_YABANCI_DESENI = re.compile(
    r"\b(is|are|the|what|which|needed|required|list|show|item|items|please)\b", re.IGNORECASE
)
NUMARALANDIRMA_DESENI = re.compile(r"^\s*\d+[\.\)\-]\s*")
SIZAN_TALIMAT_KELIMELERI_DESENI = re.compile(
    r"^\s*(tire|madde|numara|işaret)\b[:,]?\s*", re.IGNORECASE
)

PROMPT_SABLONU = '''Aşağıdaki Türkçe cümle bir KULLANICI SORUSU / BİLGİ TALEBİDİR (ör. "listeler misiniz?", "var mı?", "nedir?"). Bu cümleyi, AYNI ANLAMI ve AYNI NİYETİ koruyarak {n} farklı şekilde yeniden yaz.

KURALLAR:
- Bu bir SORU/BİLGİ TALEBİDİR — asla bir KOMUT, SİLME, KALDIRMA, DEĞİŞTİRME isteğine çevirme.
- Cümledeki olumsuzluk/gerekirlik ekini (ör. "-mesi gereken", "-meyecek", "değil", "yok") ASLA
  tersine çevirme veya kaldırma — anlam yönü birebir aynı kalmalı.
- Yalnızca TÜRKÇE yaz. Başka hiçbir dil kullanma (Çince, İngilizce vb. KESİNLİKLE YASAK).
- Cümledeki "{token}" kelimesini AYNEN, hiç değiştirmeden koru. O bir yer tutucudur —
  silme, çevirme veya başka bir kelimeyle değiştirme.
- Her varyantı ayrı satırda yaz.
- Satırların başına hiçbir işaret, sembol veya numara ekleme. Cümle doğrudan büyük
  harfle başlasın.
- Başka hiçbir açıklama, giriş cümlesi, yorum yazma — yalnızca varyantları yaz.

ÖRNEK — Cümle: KATEGORI_ADI kategorisinde kritik stok seviyesine düşen ürünleri listeler misiniz?
DOĞRU varyant (niyet ve yön aynı): KATEGORI_ADI kategorisinde stoğu kritik seviyeye inmiş ürünleri gösterir misiniz?
YANLIŞ varyant (asla böyle yapma — SORU, KOMUTA döndü): KATEGORI_ADI kategorisindeki ürünleri listeden kaldırın.

Cümle: {sablon}'''

DOGRULAMA_PROMPT = '''Aşağıdaki iki Türkçe cümle AYNI bilgiyi mi istiyor? İkisi de aynı niyeti \
(soru mu komut mu, olumlu mu olumsuz mu, "gerekli" mi "gereksiz" mi) taşımalı.

Cümle 1: {orijinal}
Cümle 2: {aday}

Yalnızca TEK KELİME cevap ver: EVET ya da HAYIR. Açıklama yazma, başka hiçbir şey ekleme.
Cevap:'''

ANLAM_ISTATISTIKLERI = {"kontrol_edilen": 0, "evet": 0, "hayir": 0, "belirsiz": 0}
"""5. hücrede toplam rapor için. `belirsiz`: model EVET/HAYIR biçimine uymadı —
bu YÜKSEK çıkarsa asıl sorun üretilen içerikte değil, DOGRULAMA_PROMPT'un
biçim takibinde demektir (gerçek Colab denemesinde tam olarak bu şüphelenildi:
%64.7 red oranı, ama örnekte kabul edilenlerin çoğu zaten doğruydu — ilk
denemede `max_new_tokens=5` modelin EVET/HAYIR'a varmadan kesilmesine yol
açıyor olabilirdi, bu yüzden hem tuş sayısı artırıldı hem ayrıştırma daha
toleranslı hâle getirildi, hem de bu ayrı sayaç eklendi)."""


def anlam_korundu_mu(orijinal: str, aday: str) -> bool:
    """İkinci, kısa bir LLM çağrısıyla niyet/anlam yönünü doğrular.

    A3.4'teki guard deseniyle aynı fikir: üreten modelin kendisine, ürettiği
    şeyin gerçekten istenen şey olup olmadığını ayrıca sor. `do_sample=False`
    (greedy) kullanılıyor — bu bir doğrulama, çeşitlilik burada istenmiyor.
    Biçimsel belirsizlik (ne EVET ne HAYIR net biçimde geçiyor) durumunda
    LENIENT davranılır (kabul edilir) — bu ek bir kalite süzgeci, A3.4'teki
    sayısal guard kadar sert bir güvenlik sınırı değil; format hatasını
    "hayır" saymak yanlışlıkla iyi içeriği eleyebilir (gerçek Colab
    denemesinde şüphelenilen tam olarak buydu).
    """
    ANLAM_ISTATISTIKLERI["kontrol_edilen"] += 1
    mesajlar = [
        {"role": "user", "content": DOGRULAMA_PROMPT.format(orijinal=orijinal, aday=aday)}
    ]
    girdi = tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True)
    girdi_id = tokenizer(girdi, return_tensors="pt").to(model.device)
    with torch.no_grad():
        cikti = model.generate(
            **girdi_id,
            max_new_tokens=16,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    yanit = tokenizer.decode(
        cikti[0][girdi_id["input_ids"].shape[1] :], skip_special_tokens=True
    ).upper()

    evet_konumu = yanit.find("EVET")
    hayir_konumu = yanit.find("HAYIR")
    if evet_konumu == -1 and hayir_konumu == -1:
        ANLAM_ISTATISTIKLERI["belirsiz"] += 1
        return True  # bkz. docstring — biçim belirsizse elemeyi tercih etme
    if hayir_konumu != -1 and (evet_konumu == -1 or hayir_konumu < evet_konumu):
        ANLAM_ISTATISTIKLERI["hayir"] += 1
        return False
    ANLAM_ISTATISTIKLERI["evet"] += 1
    return True


def parafraz_uret(
    sablon_metni: str, token, n: int = 2, max_new_tokens: int = 200, deneme: int = 2
) -> list[str]:
    token = token if isinstance(token, str) and token else None
    token_ifadesi = token if token else "(yok — bu cümlede yer tutucu yok)"
    mesajlar = [
        {
            "role": "user",
            "content": PROMPT_SABLONU.format(sablon=sablon_metni, token=token_ifadesi, n=n),
        }
    ]
    girdi = tokenizer.apply_chat_template(mesajlar, tokenize=False, add_generation_prompt=True)
    girdi_id = tokenizer(girdi, return_tensors="pt").to(model.device)

    for _ in range(deneme):
        with torch.no_grad():
            cikti = model.generate(
                **girdi_id,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.6,
                top_p=0.9,
                repetition_penalty=1.1,
                pad_token_id=tokenizer.eos_token_id,
            )
        metin = tokenizer.decode(
            cikti[0][girdi_id["input_ids"].shape[1] :], skip_special_tokens=True
        )
        satirlar = []
        for s in metin.split("\n"):
            if not s.strip():
                continue
            temiz = NUMARALANDIRMA_DESENI.sub("", s)
            temiz = SIZAN_TALIMAT_KELIMELERI_DESENI.sub("", temiz)
            temiz = temiz.strip(" -•\t")
            if temiz:
                satirlar.append(temiz)

        if token:
            satirlar = [s for s in satirlar if token in s]
        satirlar = [s for s in satirlar if not CJK_DESENI.search(s)]
        satirlar = [s for s in satirlar if not LATIN_YABANCI_DESENI.search(s)]
        satirlar = [s for s in satirlar if anlam_korundu_mu(sablon_metni, s)]

        if satirlar:
            return satirlar[:n]

    return []


## 5. Tüm şablonları işle

Yalnızca ~84 satır olduğu için örnekleme adımına gerek yok, hepsini tek seferde işleyip sonucu gözden geçiriyoruz.

In [ ]:
import json
import time

VARYANT_SAYISI = 18
"""Şablon başına istenen varyant sayısı. Seyrek araçlar (ör.
`onay_kuyrugu_sorgula`, `genel_stok_durumu_sorgula`) için sınıf ağırlıklı
örnekleme yalnızca ~10-14 örneği yüzlerce kez tekrarlamak zorunda kalır —
bu ezberletir. Çözüm veri miktarı değil çeşitlilik: bu değeri (ör. 12-15'e)
yükselterek yalnızca bu araçların şablonlarını hedefli, yüksek-n bir turdan
geçir (bkz. `--sablon-ihrac-araclar` bayrağı, modülün "Sonrası" hücresi)."""

with open(sablon_dosyasi, encoding="utf-8") as f:
    sablonlar = [json.loads(satir) for satir in f]

print("Toplam şablon sayısı:", len(sablonlar))
print("Şablon başına istenen varyant sayısı:", VARYANT_SAYISI)

t0 = time.time()
sonuclar = []
for kayit in sablonlar:
    varyantlar = parafraz_uret(
        kayit["sablon_metni"], kayit["placeholder_token"], n=VARYANT_SAYISI
    )
    sonuclar.append({**kayit, "parafraz_varyantlari": varyantlar})
t1 = time.time()

bos_sayisi = sum(1 for s in sonuclar if not s["parafraz_varyantlari"])
toplam_kontrol = ANLAM_ISTATISTIKLERI["kontrol_edilen"]
print(f"Süre: {t1 - t0:.1f} sn")
print(f"Guard tarafından reddedilen (boş kalan) şablon: {bos_sayisi}/{len(sonuclar)}")
print(f"Anlam/niyet kontrolü — toplam aday: {toplam_kontrol}")
print(
    f"  EVET (kabul): {ANLAM_ISTATISTIKLERI['evet']} "
    f"(%{100 * ANLAM_ISTATISTIKLERI['evet'] / max(1, toplam_kontrol):.1f})"
)
print(
    f"  HAYIR (red): {ANLAM_ISTATISTIKLERI['hayir']} "
    f"(%{100 * ANLAM_ISTATISTIKLERI['hayir'] / max(1, toplam_kontrol):.1f})"
)
print(
    f"  BELİRSİZ (format hatası, kabul sayıldı): {ANLAM_ISTATISTIKLERI['belirsiz']} "
    f"(%{100 * ANLAM_ISTATISTIKLERI['belirsiz'] / max(1, toplam_kontrol):.1f})"
)

ortalama_varyant = sum(len(s["parafraz_varyantlari"]) for s in sonuclar) / len(sonuclar)
print(f"Şablon başına ortalama üretilen (kabul edilen) varyant: {ortalama_varyant:.1f}")

print()
print("--- İlk 5 örnek ---")
for s in sonuclar[:5]:
    print("Orijinal:", s["sablon_metni"])
    print("Varyantlar:", s["parafraz_varyantlari"])
    print()


## 6. Kalite kontrolü (elle)

**Durup yukarıdaki örnekleri oku.** Varyantlar doğal mı, **anlam/niyet yönü
korunuyor mu** (SORU hâlâ SORU mu, olumsuzluk tersine dönmüş mü), yer tutucu
(`KATEGORI_ADI` vb.) hâlâ metinde duruyor mu? Anlam/niyet reddedilme oranı
(bir üstteki hücrenin çıktısı) hâlâ yüksekse (kabaca %20'nin üzerinde),
`DOGRULAMA_PROMPT`'un kendisi de güvenilmez demektir — `PROMPT_SABLONU`'nu
güncelleyip yeniden dene.


## 7. Sonucu kaydet ve indir

In [ ]:
from google.colab import files

CIKTI_DOSYASI = "sablon_parafraz.jsonl"

with open(CIKTI_DOSYASI, "w", encoding="utf-8") as f:
    for s in sonuclar:
        varyantlar = s["parafraz_varyantlari"] or [s["sablon_metni"]]
        for varyant in varyantlar:
            f.write(
                json.dumps(
                    {
                        "arac": s["arac"],
                        "varlik_turu": s["varlik_turu"],
                        "placeholder_token": s["placeholder_token"],
                        "sablon_metni": varyant,
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )

print("Yazıldı:", CIKTI_DOSYASI)
files.download(CIKTI_DOSYASI)


## Sonrası

**Tam turdaysan:** İndirdiğin `sablon_parafraz.jsonl` dosyasını projede
`data/egitim/sablon_parafraz.jsonl` olarak kaydet, aşağıdaki kodla tam veri
setine (gerçek ürün/kategori/tedarikçi adlarıyla) yeniden çoğalt:

```python
import pandas as pd
from training.build_dataset import parafraz_sablonlarindan_veri_uret, jsonl_yaz
from simulator.company import yapi_malzemesi_toptancisi
from simulator.catalog import katalog_uret
from pathlib import Path

parafraz_df = pd.read_json("data/egitim/sablon_parafraz.jsonl", lines=True)
kataloglar = katalog_uret(yapi_malzemesi_toptancisi(), seed=42)
tam_veri = parafraz_sablonlarindan_veri_uret(
    parafraz_df, kataloglar["sku"], kataloglar["tedarikci"]
)
jsonl_yaz(tam_veri, Path("data/egitim/router_sorulari_parafraz.jsonl"))
```

**Hedefli (seyrek araç) turdaysan:** İndirilen dosyayı
`data/egitim/sablon_parafraz_seyrek.jsonl` olarak kaydet. Bu, yalnızca 3
aracın şablonlarını (daha yüksek `n` ile üretilmiş) içeriyor — mevcut
`sablon_parafraz.jsonl` ile **birleştirilip** (bu 3 aracın eski, düşük
çeşitlilikli satırları değiştirilerek) tam veri seti yeniden üretilmeli:

```python
import pandas as pd
from training.build_dataset import parafraz_sablonlarindan_veri_uret, jsonl_yaz
from simulator.company import yapi_malzemesi_toptancisi
from simulator.catalog import katalog_uret
from pathlib import Path

SEYREK_ARACLAR = {"onay_kuyrugu_sorgula", "genel_stok_durumu_sorgula", "gecelik_ozet_sorgula"}

mevcut = pd.read_json("data/egitim/sablon_parafraz.jsonl", lines=True)
mevcut = mevcut[~mevcut["arac"].isin(SEYREK_ARACLAR)]  # eski dusuk-cesitlilik satirlarini at
yeni = pd.read_json("data/egitim/sablon_parafraz_seyrek.jsonl", lines=True)
birlesik = pd.concat([mevcut, yeni], ignore_index=True)
jsonl_yaz(birlesik, Path("data/egitim/sablon_parafraz.jsonl"))  # ustune yaz

kataloglar = katalog_uret(yapi_malzemesi_toptancisi(), seed=42)
tam_veri = parafraz_sablonlarindan_veri_uret(
    birlesik, kataloglar["sku"], kataloglar["tedarikci"]
)
jsonl_yaz(tam_veri, Path("data/egitim/router_sorulari_parafraz.jsonl"))
```

Her iki durumda da sonrasında `uv run python -m training.veri_bolme`
çalıştırılıp `router_train/val/test.jsonl` ve `golden_set_aday.jsonl`
yeniden üretilmeli.
